In [1]:
import torch
import torch.nn.functional as F

# 이미지 3개에 대한 클래스별 logit
# 클래스 순서 0=cat, 1=chicken, 2=dog
logits = torch.tensor([
    [2.0, 1.0, 0.0], # cat
    [0.0, 2.0, 1.0], # chicken
    [1.0, 0.0, 2.0], # dog
])

# 각 이미지의 실제 정답 클래스
labels = torch.tensor([0, 1, 2])

one_hot_labels = F.one_hot(
    labels,
    num_classes=3,
).to(torch.float32)

print("Logits:")
print(logits)

print("\nOne-hot labels:")
print(one_hot_labels)

Logits:
tensor([[2., 1., 0.],
        [0., 2., 1.],
        [1., 0., 2.]])

One-hot labels:
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])


In [2]:
# logit을 softmax에 통과시켜 클래스별 확률로 변환한다.
probabilities = torch.softmax(
    logits,
    dim=-1,
)

# cross-entropy:
# 정답 클래스의 예측 확률만 원-핫 인코딩으로 선택한 뒤, 
# 음의 로그 (-log) 적용
per_example_loss = - (
    one_hot_labels
    * probabilities.log()
).sum(dim=-1)

mean_loss = per_example_loss.mean()

print("Probabilities:")
print(probabilities)

print("\nProbability sums:")
print(probabilities.sum(dim=-1))

print("\nLoss per example:")
print(per_example_loss)

print("\nMean loss:")
print(mean_loss)

Probabilities:
tensor([[0.6652, 0.2447, 0.0900],
        [0.0900, 0.6652, 0.2447],
        [0.2447, 0.0900, 0.6652]])

Probability sums:
tensor([1.0000, 1.0000, 1.0000])

Loss per example:
tensor([0.4076, 0.4076, 0.4076])

Mean loss:
tensor(0.4076)


In [3]:
# (다른 방법)
# 정수 label을 이용해 정답 클래스의 확률을 직접 선택한다.
batch_indices = torch.arange(labels.shape[0])

# 아래 처럼 동작
# [
#     probabilities[0, 0],
#     probabilities[1, 1],
#     probabilities[2, 2]
# ]
correct_class_probabilities = probabilities[
    batch_indices, # tensor([0, 1, 2])
    labels,        # tensor([0, 1, 2])
] # tensor([0.6652, 0.6652, 0.6652])

# 음의 로그: Negative log
loss_from_correct_class = -correct_class_probabilities.log()

# PyTorch는 softmax 전의 logits와 정수 labels를 받는다.
pytorch_loss = F.cross_entropy(
    logits,
    labels,
    reduction="none",
)

print("Correct-class probabilities:")
print(correct_class_probabilities)

print("\nLoss from correct classes:")
print(loss_from_correct_class)

print("\nPyTorch cross-entropy:")
print(pytorch_loss)


Correct-class probabilities:
tensor([0.6652, 0.6652, 0.6652])

Loss from correct classes:
tensor([0.4076, 0.4076, 0.4076])

PyTorch cross-entropy:
tensor([0.4076, 0.4076, 0.4076])


In [4]:
# logit에 대한 cross-entropy gradient를 확인
logits_for_gradient = (
    logits
    .clone()
    .detach()
    .requires_grad_()
)

loss = F.cross_entropy(
    logits_for_gradient,
    labels,
)

loss.backward()

# 데이터 하나의 gradient는 
# softmax(logits) - one_hot_label이다.
#
# 현재 loss는 배치 평균이므로 batch_size로 나눈다.
expected_gradient = (
    torch.softmax(
        logits_for_gradient.detach(),
        dim=-1
    )
    - one_hot_labels
) / labels.shape[0]

print("Gradient from backward:")
print(logits_for_gradient.grad)

print("\nExpected gradient:")
print(expected_gradient)

Gradient from backward:
tensor([[-0.1116,  0.0816,  0.0300],
        [ 0.0300, -0.1116,  0.0816],
        [ 0.0816,  0.0300, -0.1116]])

Expected gradient:
tensor([[-0.1116,  0.0816,  0.0300],
        [ 0.0300, -0.1116,  0.0816],
        [ 0.0816,  0.0300, -0.1116]])


In [5]:
# 정답이 하나의 클래스가 아니라 확률분포인 경우
soft_labels = torch.tensor([
    [0.8, 0.1, 0.1],
    [0.1, 0.7, 0.2],
    [0.2, 0.2, 0.6],
])

manual_soft_label_loss = -(
    soft_labels
    * F.log_softmax(logits, dim=-1)
).sum(dim=-1)

pytorch_soft_label_loss = F.cross_entropy(
    logits,
    soft_labels,
    reduction="none",
)

print("Soft-label loss:")
print(manual_soft_label_loss)

assert torch.allclose(
    manual_soft_label_loss,
    pytorch_soft_label_loss,
)

Soft-label loss:
tensor([0.7076, 0.8076, 1.0076])
